# Package installation

In [49]:
#%pip install numpy
#%pip install pillow
#%pip install matplotlib
#%pip install tensorflow
#%pip install mpmath
%pip install galois

   ---------------------------------------- 0.0/4.2 MB ? eta -:--:--
   ------------------- -------------------- 2.1/4.2 MB 16.3 MB/s eta 0:00:01
   ---------------------------------------- 4.2/4.2 MB 12.7 MB/s  0:00:00
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 2.8/2.8 MB 87.8 MB/s  0:00:00
   ---------------------------------------- 0.0/38.1 MB ? eta -:--:--
   ------------------------- -------------- 24.4/38.1 MB 116.5 MB/s eta 0:00:01
   ---------------------------------------- 38.1/38.1 MB 106.9 MB/s  0:00:00

   ---------------------------------------- 0/3 [llvmlite]
   ---------------------------------------- 0/3 [llvmlite]
   ---------------------------------------- 0/3 [llvmlite]
   ---------------------------------------- 0/3 [llvmlite]
   ------------- -------------------------- 1/3 [numba]
   ------------- -------------------------- 1/3 [numba]
   ------------- -------------------------- 1/3 [numba]
   --

In [50]:
import numpy as np
from PIL import Image
from itertools import permutations
from typing import List
from itertools import permutations
import matplotlib.pyplot as plt
import secrets
import galois

# ALgorith preparation

## KEYS

In [53]:
def generar_llave(n: int) -> List[List[int]]:
    """
    Genera una llave de tamaño n (n % 3 == 0),
    aplica DTQW global, luego divide en 3 partes iguales
    y aplica DTQW nuevamente a cada segmento.
    Retorna un array de longitud 3.
    """

    if n < 15 or n % 3 != 0:
        raise ValueError("n debe ser >= 15 y divisible por 3")

    seed = np.random.SeedSequence().entropy
    rng = np.random.default_rng(seed)

    def generar_aleatoria():
        return rng.integers(0, 24, size=n).tolist()

    def coin_matrix(theta):
        return np.array(
            [[np.cos(theta), np.sin(theta)], [np.sin(theta), -np.cos(theta)]]
        )

    def shift(psi):
        psi_new = np.zeros_like(psi)
        psi_new[0, :] = np.roll(psi[0, :], -1)
        psi_new[1, :] = np.roll(psi[1, :], 1)
        return psi_new

    def reordenar(K):
        m = len(K)

        suma = K[0] + K[1] + K[2]
        theta1 = (K[0] / suma) * (np.pi / 2)
        theta2 = (K[1] / suma) * (np.pi / 2)
        theta3 = (K[2] / suma) * (np.pi / 2)

        m_bits = [int(b) for b in format(K[3], "08b")]
        r_pasos = K[3] + K[4]

        psi = np.zeros((2, m), dtype=complex)
        psi[0, 0] = 1.0

        for t in range(r_pasos):
            if t < len(m_bits):
                theta = theta1 if m_bits[t] == 0 else theta2
            else:
                theta = theta3

            C = coin_matrix(theta)
            psi_after = np.zeros_like(psi)

            for j in range(m):
                psi_after[:, j] = C @ psi[:, j]

            psi = shift(psi_after)

        probs = np.abs(psi[0]) ** 2 + np.abs(psi[1]) ** 2
        orden = np.argsort(probs)[::-1]

        return [K[i] for i in orden]

    # 1. Generar llave base
    K = generar_aleatoria()

    # 2. DTQW global
    K_global = reordenar(K)

    # 3. Dividir en 3 partes
    size = n // 3
    K1 = K_global[:size]
    K2 = K_global[size : 2 * size]
    K3 = K_global[2 * size :]

    # 4. DTQW por segmento
    K1 = reordenar(K1)
    K2 = reordenar(K2)
    K3 = reordenar(K3)

    # 5. Retorno
    return [K1, K2, K3]

## Image

In [54]:
def visualizar_canal_color(canal_img: Image.Image, color: str) -> Image.Image:
    color = color.upper()
    if color not in ["R", "G", "B"]:
        raise ValueError()

    arr = np.array(canal_img)
    zeros = np.zeros_like(arr)

    if color == "R":
        rgb = np.stack([arr, zeros, zeros], axis=2)
    elif color == "G":
        rgb = np.stack([zeros, arr, zeros], axis=2)
    else:
        rgb = np.stack([zeros, zeros, arr], axis=2)

    return Image.fromarray(rgb, mode="RGB")


def obtener_rgb(ruta_png: str):
    img = Image.open(ruta_png).convert("RGB")
    r_img, g_img, b_img = img.split()
    return r_img, g_img, b_img

## Cipher

In [ ]:
GF = galois.GF(256)


def generar_matriz_invertible_gf(dim: int, seed_val: int) -> galois.FieldArray:
    """
    Reemplaza a generar_matriz_mezcla (que usaba QR y floats).
    Genera una matriz aleatoria en GF(256) garantizando que sea invertible (Determinante != 0).
    """
    rng = np.random.default_rng(seed_val)
    while True:
        M_np = rng.integers(0, 256, (dim, dim))
        M_gf = GF(M_np)
        if np.linalg.det(M_gf) != 0:
            return M_gf


def canal_a_bloques_gf(canal_arr: np.ndarray) -> galois.FieldArray:
    """Convierte un canal plano a bloques 4xN en formato Galois."""
    arr = canal_arr.flatten()
    pad = (4 - len(arr) % 4) % 4
    if pad:
        arr = np.concatenate([arr, np.zeros(pad, dtype=arr.dtype)])
    return GF(arr.reshape(4, -1))


def obtener_rgb_como_bloques_gf(R, G, B):
    return canal_a_bloques_gf(R), canal_a_bloques_gf(G), canal_a_bloques_gf(B)


def cifrar_canal_gf(canal, matrices_finales: list) -> galois.FieldArray:
    """Aplica las matrices de cifrado. Forzamos GF() para evitar incompatibilidades."""
    resultado = GF(canal)
    for Mi in matrices_finales:
        resultado = GF(Mi) @ resultado
    return resultado


def mezclar_canales_gf(R_b, G_b, B_b, M_mix):
    """Mezcla los tres canales. Las envolturas GF() previenen el TypeError."""
    stack = GF(np.stack([np.asarray(R_b), np.asarray(G_b), np.asarray(B_b)], axis=0))
    stack = stack.reshape(3, -1)

    # Multiplicación estricta en GF
    mezclado = GF(M_mix) @ stack
    mezclado = mezclado.reshape(3, 4, -1)
    return GF(mezclado[0]), GF(mezclado[1]), GF(mezclado[2])


def permutacion_global_pixeles(canal_2d: np.ndarray, seed_val: int) -> np.ndarray:
    """Destruye la correlación adyacente reordenando linealmente."""
    rng = np.random.default_rng(seed_val)
    h, w = canal_2d.shape
    canal_plano = canal_2d.flatten()
    idx = rng.permutation(h * w)
    return canal_plano[idx].reshape((h, w))


def construir_llave_permutacion_gf(keys: list) -> galois.FieldArray:
    K1, K2, K3 = GF(keys[0]), GF(keys[1]), GF(keys[2])
    return K1 + K2 + K3


def rotar_columnas(imagen: np.ndarray, K: np.ndarray) -> np.ndarray:
    """(Queda igual, solo mueve posiciones)"""
    resultado = imagen.copy()
    h_img, w_img = imagen.shape
    n = len(K)
    for i in range(w_img):
        # np.asarray asegura que si K viene de GF, se trate como entero normal para el índice
        desplazamiento = int(np.asarray(K[i % n])) % h_img
        resultado[:, i] = np.roll(imagen[:, i], desplazamiento)
    return resultado


def permutacion_filas_columnas(imagen: np.ndarray, K: np.ndarray) -> np.ndarray:
    """(Queda igual, aunque te recomiendo usar permutacion_global_pixeles para más seguridad)"""
    img_col = rotar_columnas(imagen, K)
    img_fila = rotar_columnas(img_col.T, K).T
    return img_fila


def generar_matrices_4x4_gf(seed_val: int) -> list:
    """Genera las matrices de cifrado para cada canal usando Galois."""
    rng = np.random.default_rng(seed_val)
    seeds = rng.integers(0, 2**32 - 1, size=(3, 2))

    matrices_sets = []
    for c in range(3):
        matrices_canal = [generar_matriz_invertible_gf(4, int(s)) for s in seeds[c]]
        matrices_sets.append(matrices_canal)
    return matrices_sets


def matrices_encriptacion(llave: List[int], matrices_rotacion):
    matrices_encript = []
    for item in llave:
        matrices_encript.append(matrices_rotacion[item % 24])
    return matrices_encript

# Cipher Implementation Algorithm

In [ ]:
# ---------------------------------------------------------
# 4. PIPELINE DE EJECUCIÓN PRINCIPAL
# ---------------------------------------------------------
path = r"./Imagenes/4.2.03.png"
img = Image.open(path).convert("RGB")
R, G, B = img.split()
h, w = img.height, img.width

R_arr = np.array(R)
G_arr = np.array(G)
B_arr = np.array(B)

seed = secrets.randbits(32)

keys = [
    np.random.default_rng(seed + 1).integers(0, 256, max(h, w)),
    np.random.default_rng(seed + 2).integers(0, 256, max(h, w)),
    np.random.default_rng(seed + 3).integers(0, 256, max(h, w)),
]

matrices_sets = generar_matrices_4x4_gf(seed)

# --- ETAPA A ---
R_arr_p = permutacion_global_pixeles(R_arr, seed + 10)
G_arr_p = permutacion_global_pixeles(G_arr, seed + 20)
B_arr_p = permutacion_global_pixeles(B_arr, seed + 30)

# --- ETAPA B ---
R_b = canal_a_bloques_gf(R_arr_p)
G_b = canal_a_bloques_gf(G_arr_p)
B_b = canal_a_bloques_gf(B_arr_p)

# --- ETAPA C ---
M_mix = generar_matriz_invertible_gf(3, seed + 99)
R_m, G_m, B_m = mezclar_canales_gf(R_b, G_b, B_b, M_mix)

# --- ETAPA D ---
c_r = cifrar_canal_gf(R_m, matrices_sets[0])
c_g = cifrar_canal_gf(G_m, matrices_sets[1])
c_b = cifrar_canal_gf(B_m, matrices_sets[2])

# Reconstrucción 2D
c_r_2d = np.asarray(c_r).reshape(-1)[: h * w].reshape(h, w)
c_g_2d = np.asarray(c_g).reshape(-1)[: h * w].reshape(h, w)
c_b_2d = np.asarray(c_b).reshape(-1)[: h * w].reshape(h, w)

# --- ETAPA E ---
K_perm = np.asarray(construir_llave_permutacion_gf(keys))

r_final = permutacion_global_pixeles(c_r_2d, seed + 100)
g_final = permutacion_global_pixeles(c_g_2d, seed + 200)
b_final = permutacion_global_pixeles(c_b_2d, seed + 300)

# --- GUARDADO ---
cifrado_visual = np.stack([r_final, g_final, b_final], axis=2).astype(np.uint8)
ruta_guardado = r"./Imagenes/cifrado.png"
Image.fromarray(cifrado_visual, mode="RGB").save(ruta_guardado)

print("¡Cifrado completado exitosamente con GF(2^8)!")

RED [array([[ 63., 122.,  50.,  73.],
       [ 20.,  24.,  41.,  62.],
       [ 96.,  39.,  76.,  78.],
       [ 10., 110.,  42.,  39.]]), array([[ 63., 122.,  50.,  73.],
       [ 20.,  24.,  41.,  62.],
       [ 76.,  78.,  96.,  39.],
       [ 42.,  39.,  10., 110.]]), array([[ 63., 122.,  96.,  39.],
       [ 20.,  24.,  10., 110.],
       [ 50.,  73.,  76.,  78.],
       [ 41.,  62.,  42.,  39.]]), array([[ 63., 122.,  96.,  39.],
       [ 20.,  24.,  10., 110.],
       [ 76.,  78.,  50.,  73.],
       [ 42.,  39.,  41.,  62.]]), array([[ 63., 122.,  76.,  78.],
       [ 20.,  24.,  42.,  39.],
       [ 50.,  73.,  96.,  39.],
       [ 41.,  62.,  10., 110.]]), array([[ 63., 122.,  76.,  78.],
       [ 20.,  24.,  42.,  39.],
       [ 96.,  39.,  50.,  73.],
       [ 10., 110.,  41.,  62.]]), array([[ 50.,  73.,  63., 122.],
       [ 41.,  62.,  20.,  24.],
       [ 96.,  39.,  76.,  78.],
       [ 10., 110.,  42.,  39.]]), array([[ 50.,  73.,  63., 122.],
       [ 41.,  62.,  20.

TypeError: Argument 'A' must be an instance of <class 'galois.GF(2^8, primitive_element='x', irreducible_poly='x^8 + x^4 + x^3 + x^2 + 1')'>, not <class 'numpy.ndarray'>.

# CHANGES 

This involves swapping those deterministic parts into quantum - chaotic ones

## New Matrix Gens

I'll use Quantum Logistic Chaotic Map, for selecting the keys

CITE
Phys. Rev. A 41, 5705(R) – Published 1 May, 1990

DOI: https://doi.org/10.1103/PhysRevA.41.5705

$$
\begin{aligned}
x_{n+1} &= \eta(x_n - |x_n|^2) - \eta y_n \\[10pt]
y_{n+1} &= -y_n e^{-2\gamma} + e^{-\gamma}\eta \left[ (2 - x_n - x_n^*)y_n - x_n z_n^* - x_n^* z_n \right] \\[10pt]
z_{n+1} &= -z_n e^{-2\gamma} + e^{-\gamma}\eta \left[ 2(1 - x_n^*)z_n - 2x_n y_n - x_n \right]
\end{aligned}
$$

In [10]:
import mpmath as mp

# ---------------------------------------------------------
# PRECISION SETTINGS
# ---------------------------------------------------------
# Set the exact number of decimal places you want here.
# You can set this to 50, 100, 1000, or even more.
mp.mp.dps = 100


def simulate_3d_qplm_high_precision(x, y , z, iterations=1500):
    # Initialize parameters as high-precision floats
    eta = mp.mpf("4.0")
    gamma = mp.mpf("6.0")

    # Precompute exponential constants with high precision
    exp_minus_gamma = mp.exp(-gamma)
    exp_minus_2gamma = mp.exp(-2 * gamma)

    # ---------------------------------------------------------
    # INITIAL CONDITIONS
    # ---------------------------------------------------------
    x0_real = 0.3 + (x % 7000) / 10000.0
    y0_real = 0.05 + ((y // 7000) % 500) / 10000.0
    z0_real = 0.1 + ((z // 3500000) % 1000) / 10000.0

    # mp.mpc creates a high-precision complex number
    x = mp.mpc(x0_real, 0)
    y = mp.mpc(y0_real, 0)
    z = mp.mpc(z0_real, 0)

    history = [(x, y, z)]

    # Constants used in equations
    two = mp.mpf("2.0")
    one = mp.mpf("1.0")

    # ---------------------------------------------------------
    # EXECUTION
    # ---------------------------------------------------------
    for i in range(1, iterations + 1):
        # Magnitudes and conjugates
        x_abs_sq = x.real**2 + x.imag**2
        x_conj = mp.conj(x)
        z_conj = mp.conj(z)

        # x_{n+1}
        x_next = eta * (x - x_abs_sq) - eta * y

        # y_{n+1}
        term_y = (two - x - x_conj) * y - x * z_conj - x_conj * z
        y_next = -y * exp_minus_2gamma + exp_minus_gamma * eta * term_y

        # z_{n+1}
        term_z = two * (one - x_conj) * z - two * x * y - x
        z_next = -z * exp_minus_2gamma + exp_minus_gamma * eta * term_z

        # ---------------------------------------------------------
        # VALIDATIONS
        # ---------------------------------------------------------
        # mpmath handles massive numbers smoothly, but we still check for absolute divergence
        if (
            mp.isnan(x_next.real)
            or mp.isinf(x_next.real)
            or mp.isnan(y_next.real)
            or mp.isinf(y_next.real)
            or mp.isnan(z_next.real)
            or mp.isinf(z_next.real)
        ):
            raise ValueError(
                f"Divergence error: System reached Inf or NaN at iteration {i}"
            )

        x, y, z = x_next, y_next, z_next
        history.append((x, y, z))

    return history


def format_mpmath_complex(c, decimals):
    """Formats high-precision complex numbers to the requested decimal count."""
    # mpmath has built-in string conversion that respects the global dps setting
    sign = "+" if c.imag >= 0 else "-"
    # Convert exactly to string using nstr
    real_str = mp.nstr(c.real, n=decimals)
    imag_str = mp.nstr(abs(c.imag), n=decimals)
    return f"{real_str} {sign} {imag_str}i"


if __name__ == "__main__":
    TARGET_ITERATION = 1500
    X = 23
    Y = 91
    Z = 567
    
    try:
        trajectory = simulate_3d_qplm_high_precision(
            x=X,y=Y,z=Z, iterations=TARGET_ITERATION
        )
        final_x, final_y, final_z = trajectory[TARGET_ITERATION]

        output_str = (
            f"Iteration {TARGET_ITERATION} → \n"
            f"x = {format_mpmath_complex(final_x, mp.mp.dps)}\n"
            f"y = {format_mpmath_complex(final_y, mp.mp.dps)}\n"
            f"z = {format_mpmath_complex(final_z, mp.mp.dps)}"
        )
        print(output_str)

    except ValueError as e:
        print(e)

Iteration 1500 → 
x = 0.7226002622077230322900592562452334054541701461423756395132987116691226903382865160021447977203703625 + 0.0i
y = 0.00004468292898913844195557113330070736665654601403029505477466575130882685405322857751540578010820754845 + 0.0i
z = -0.002488667167152985251416678426287003977098309179047781889518744860709382026707271935412392213539961754 + 0.0i
